# README

Notebook que treina e testa o modelo já reduzido

In [1]:
import torch
print("CUDA disponível:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("Nome da GPU:", torch.cuda.get_device_name(0))
    print("Versão CUDA do PyTorch:", torch.version.cuda)

CUDA disponível: True
Nome da GPU: NVIDIA GeForce GTX 1650
Versão CUDA do PyTorch: 12.1


In [2]:
dimension = [8,16]

# Arquitetura 
As entradas foram reduzidas para 8x16. 
É uma rede simples, com uma camada fully connected. 


## Carregar o dataset

In [3]:
from torchvision import datasets, transforms
import os

transform = transforms.Compose([
    transforms.Grayscale(),     
    transforms.Resize((75, 100)),
    transforms.ToTensor()
])

!ls CNN_letter_Dataset/test

test_data  = datasets.ImageFolder(root="CNN_letter_Dataset/test", transform=transform)
train_data  = datasets.ImageFolder(root="CNN_letter_Dataset/train", transform=transform)

# --- DataLoaders ---
train_loader = torch.utils.data.DataLoader(train_data, batch_size=10, shuffle=True)
test_loader  = torch.utils.data.DataLoader(test_data, batch_size=10, shuffle=False)

# --- Informações básicas ---
print(f"Total de imagens de treino: {len(train_data)}")
print(f"Total de imagens de teste:  {len(test_data)}")

0  2  4  6  8  A  C  E	G  I  K  M  P  R  T  V	X  Z
1  3  5  7  9  B  D  F	H  J  L  N  Q  S  U  W	Y
Total de imagens de treino: 28400
Total de imagens de teste:  7100


### Normalizar o dataset

In [4]:
import torch.nn.functional as F

interpolate = True
important = False
def interpolate_image(img_tensor, new_dim=dimension):
    """
    Recebe uma imagem [C,H,W] ou [H,W] e retorna vetor [num_top_pixels]
    contendo apenas os pixels mais importantes no novo espaço.
    """
    # Remove canal se necessário
    img = img_tensor.squeeze()  # [H, W]
    img_resized = F.interpolate(
        img.unsqueeze(0).unsqueeze(0),  # adiciona batch e canal
        size=(new_dim[0], new_dim[1]),
        mode='bilinear',
        align_corners=False
    ).squeeze()  # remove batch e canal

    return img_resized.flatten()  

In [5]:
from torch.utils.data import TensorDataset, DataLoader
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Reduz conjunto de treino
#X_train_reduced = torch.stack([reduce_image(img, top_pixel_coordinates_int) for img, _ in train_data])
X_train_reduced = torch.stack([interpolate_image(img) for img, _ in train_data])
y_train = torch.tensor([label for _, label in train_data])


# Reduz conjunto de teste
X_test_reduced = torch.stack([interpolate_image(img) for img, _ in test_data])
y_test = torch.tensor([label for _, label in test_data])

print(f"Novo shape de treino: {X_train_reduced.shape}")
print(f"Novo shape de teste:  {X_test_reduced.shape}")

# DataLoaders
batch_size = 10
train_loader_reduced = DataLoader(TensorDataset(X_train_reduced, y_train), batch_size=batch_size, shuffle=True)
test_loader_reduced = DataLoader(TensorDataset(X_test_reduced, y_test), batch_size=batch_size)


Novo shape de treino: torch.Size([28400, 128])
Novo shape de teste:  torch.Size([7100, 128])


## Modelo

In [6]:
import torch.nn as nn

num_pixels = dimension[0] * dimension[1]  
n_classes = len(train_data.classes)    

model_small = nn.Sequential(
    nn.Linear(num_pixels, n_classes)
)


## Treinamento

In [7]:
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#device = "cpu"
model_small.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model_small.parameters(), lr=0.1)

epochs = 5
losses_interpolate = []

for epoch in range(epochs):
    print(f'Epoch: {epoch}')
    model_small.train()
    total_loss = 0
    
    for X_batch, y_batch in tqdm(train_loader_reduced):
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        output = model_small(X_batch)
        loss = criterion(output, y_batch)
        total_loss += loss.item()
        loss.backward()
        optimizer.step()
    

    losses_interpolate.append(total_loss / len(train_loader_reduced))
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader_reduced):.4f}")



Epoch: 0


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2840/2840 [00:27<00:00, 102.76it/s]


Epoch 1, Loss: 1.7645
Epoch: 1


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2840/2840 [00:04<00:00, 675.25it/s]


Epoch 2, Loss: 0.8899
Epoch: 2


...

Epoch 3, Loss: 0.6962
Epoch: 3


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2840/2840 [00:07<00:00, 396.48it/s]


Epoch 4, Loss: 0.6008
Epoch: 4


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2840/2840 [00:05<00:00, 544.49it/s]

Epoch 5, Loss: 0.5397


In [8]:
print(losses_interpolate)

[1.7644912992144974, 0.889920267247608, 0.6962428448748, 0.6007933707330638, 0.5397033619602591]


## Teste

In [9]:
from torch.utils.data import TensorDataset, DataLoader
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Lista de imagens reduzidas usando as coordenadas corretas
X_test_reduced = torch.stack([interpolate_image(img) for img, _ in test_data])
y_test = torch.tensor([label for _, label in test_data])

print(f"Shape do dataset reduzido: {X_test_reduced.shape}")  # deve ser (N_test, num_top_pixels)

# DataLoader para o modelo reduzido
test_loader_reduced = DataLoader(
    TensorDataset(X_test_reduced, y_test),
    batch_size=10,  # ou outro batch_size que quiser
    shuffle=False
)

Shape do dataset reduzido: torch.Size([7100, 128])


In [10]:
def calc_loss_reduced(model, criterion, loader):
    """
    Calcula a loss média de um modelo MLP que recebe entradas já achatadas.
    loader: DataLoader do dataset reduzido (ex: test_loader_reduced)
    """
    batch_losses = []
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model.eval()
    device = next(model.parameters()).device

    with torch.no_grad():
        for img_batch, label_batch in loader:
            img_batch = img_batch.to(device)
            label_batch = label_batch.to(device)

            output = model(img_batch)
            loss = criterion(output, label_batch).item()
            batch_losses.append(loss)

    mean_loss = sum(batch_losses) / len(batch_losses)
    return batch_losses, mean_loss


In [11]:
batch_losses_interpolate, mean_loss_interpolate = calc_loss_reduced(model_small, criterion, test_loader_reduced)
print(f"Loss média: {mean_loss_interpolate:.4f}")
print(f"Loss por batch (primeiros 5): {batch_losses_interpolate[:10]}")


Loss média: 0.5276
Loss por batch (primeiros 5): [0.611914873123169, 0.8468480110168457, 0.9767094850540161, 0.3249250054359436, 1.8647072315216064, 0.6701943874359131, 0.7654481530189514, 0.8811214566230774, 0.49683889746665955, 0.7247675657272339]


In [12]:
def calc_accuracy_reduced(model, loader, n_classes=None):
    """
    Calcula acurácia geral e por classe de um modelo MLP.
    loader: DataLoader do dataset reduzido.
    n_classes: número de classes (se None, inferido do loader).
    """
    if n_classes is None:
        n_classes = len(loader.dataset.tensors[1].unique())
    
    correct_total = 0
    total_total = 0
    correct_per_class = {i: 0 for i in range(n_classes)}
    total_per_class = {i: 0 for i in range(n_classes)}
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model.eval()
    device = next(model.parameters()).device

    with torch.no_grad():
        for img_batch, label_batch in loader:
            img_batch = img_batch.to(device)
            label_batch = label_batch.to(device)

            output = model(img_batch)
            preds = output.argmax(dim=1)

            correct_total += (preds == label_batch).sum().item()
            total_total += len(label_batch)

            for i in range(len(label_batch)):
                label = label_batch[i].item()
                total_per_class[label] += 1
                if preds[i] == label_batch[i]:
                    correct_per_class[label] += 1

    class_acc = {cls: correct_per_class[cls]/total_per_class[cls] 
                 if total_per_class[cls] > 0 else 0
                 for cls in range(n_classes)}

    overall_acc = correct_total / total_total
    return overall_acc, class_acc


In [13]:
overall_acc_interpolate, class_acc_interpolate = calc_accuracy_reduced(model_small, test_loader_reduced)
print(f"Acurácia geral: {overall_acc_interpolate*100:.2f}%")
for cls, acc in class_acc_interpolate.items():
    print(f"Classe {cls}: {acc*100:.2f}%")

Acurácia geral: 88.66%
Classe 0: 76.70%
Classe 1: 95.15%
Classe 2: 90.29%
Classe 3: 88.35%
Classe 4: 94.66%
Classe 5: 96.60%
Classe 6: 84.47%
Classe 7: 83.98%
Classe 8: 69.90%
Classe 9: 87.86%
Classe 10: 87.62%
Classe 11: 83.98%
Classe 12: 97.55%
Classe 13: 92.08%
Classe 14: 97.52%
Classe 15: 92.65%
Classe 16: 97.55%
Classe 17: 89.71%
Classe 18: 52.97%
Classe 19: 97.09%
Classe 20: 96.53%
Classe 21: 93.56%
Classe 22: 95.10%
Classe 23: 91.18%
Classe 24: 94.55%
Classe 25: 72.28%
Classe 26: 91.67%
Classe 27: 93.63%
Classe 28: 88.24%
Classe 29: 83.17%
Classe 30: 84.95%
Classe 31: 95.54%
Classe 32: 91.09%
Classe 33: 77.72%
Classe 34: 99.38%
